In [ ]:
# ===== NB13 | CELL 1: header =====
from google.colab import drive
drive.mount('/content/drive')
print('CELL 1 DONE')

In [ ]:
# ===== NB13 | CELL 2: write runlib.py =====
# WHAT: a helper that loads the final system (any list of runs) and predicts probabilities for NEW texts.
# WHY:  the fairness and LIME notebooks need to score new sentences with the exact frozen system.
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
CODE = r'''"""Load the frozen model (any list of runs) and predict probabilities for new texts."""
import numpy as np, pandas as pd

CKPT_NAMES = {'B1_256': 'B1_distilbert_bce_best', 'B2_256': 'B2_roberta_bce_best'}


class Ensemble:
    def __init__(self, proj, runs, batch_size=64):
        import joblib
        self.runs, self.bs, self.items = list(runs), batch_size, []
        reg = pd.read_csv(f'{proj}/results/experiment_logs/registry.csv').set_index('run_id')
        ck = f'{proj}/models/saved_checkpoints'
        for r in self.runs:
            if r in ('A1', 'A2'):
                word = joblib.load(f'{ck}/word_tfidf_vectorizer.pkl')
                char = joblib.load(f'{ck}/char_tfidf_vectorizer.pkl')
                fn = 'A1_logistic_regression' if r == 'A1' else 'A2_calibrated_linear_svm'
                self.items.append(('classical', r, (word, char, joblib.load(f'{ck}/{fn}.pkl'))))
            else:
                import torch
                from transformers import AutoTokenizer, AutoModelForSequenceClassification
                d = f'{ck}/{CKPT_NAMES.get(r, r + "_best")}'
                tok = AutoTokenizer.from_pretrained(d)
                dev = 'cuda' if torch.cuda.is_available() else 'cpu'
                model = AutoModelForSequenceClassification.from_pretrained(d).to(dev).eval()
                self.items.append(('neural', r, (tok, model, int(reg.loc[r, 'max_len']))))
        print('Loaded models:', [(k, r) for k, r, _ in self.items])

    def predict(self, texts):
        import scipy.sparse as sp
        texts = [t if isinstance(t, str) else '' for t in texts]
        outs = []
        for kind, r, obj in self.items:
            if kind == 'classical':
                word, char, clf = obj
                parts = []
                for s in range(0, len(texts), 8000):
                    chunk = texts[s:s + 8000]
                    X = sp.hstack([word.transform(chunk), char.transform(chunk)]).tocsr()
                    parts.append(clf.predict_proba(X))
                outs.append(np.vstack(parts))
            else:
                from nnlib import predict_with
                tok, model, ml = obj
                outs.append(predict_with(model, tok, texts, ml, self.bs))
        return np.mean(outs, axis=0)

    def free(self):
        self.items = []
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
'''
import sys, importlib
open(f'{PROJECT_DIR}/src/runlib.py', 'w').write(CODE)
sys.path.insert(0, f'{PROJECT_DIR}/src')
import runlib; importlib.reload(runlib)
print('runlib written and imported.')
print('CELL 2 DONE')

In [ ]:
# ===== NB13 | CELL 3: ensemble selection + thresholds + FREEZE (validation data only) =====
# WHAT: average the predictions of the best runs in every combination of 1 to 3, pick the best using
#       validation Macro AP, tune one threshold per label on validation, and save the frozen configuration.
# WHY:  van Aken et al. showed that models with different errors work better together. The rule is fixed in
#       advance: the simplest candidate within 0.002 AP of the best, and it must contain a DistilBERT run
#       (the brief). The test sets are NOT used here.
import sys, importlib, json, itertools, os, time
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, load_split, load_probs, macro_ap, tune_thresholds, per_label_report
from sklearn.metrics import f1_score
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
for d in ['graphs', 'tables', 'final']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)

Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
reg = pd.read_csv(f'{PROJECT_DIR}/results/experiment_logs/registry.csv')
runs = [r for r in reg.run_id if os.path.exists(f'{PROJECT_DIR}/results/predictions/{r}_val.npy')]
P = {r: load_probs(PROJECT_DIR, r, 'val') for r in runs}
ap = {r: macro_ap(Yva, P[r]) for r in runs}
print('Validation Macro AP of every run:'); print(pd.Series(ap).round(4).sort_values(ascending=False).to_string())

top = sorted(runs, key=lambda r: -ap[r])[:5]
DISTIL = {'B1_256', 'B1_128', 'C1_128', 'C2_128'}          # the brief requires DistilBERT in the final system
combos = [c for k in (1, 2, 3) for c in itertools.combinations(top, k) if set(c) & DISTIL]
if not combos:
    combos = [c for k in (1, 2, 3) for c in itertools.combinations(top, k)]
rows = []
for c in combos:
    Pc = np.mean([P[r] for r in c], axis=0)
    thr = tune_thresholds(Yva, Pc); rep = per_label_report(Yva, Pc, thr)
    rows.append({'runs': '+'.join(c), 'n_models': len(c), 'val_macro_AP': round(macro_ap(Yva, Pc), 4),
                 'val_macro_F1_tuned': round(rep.F1.mean(), 4)})
cand = pd.DataFrame(rows).sort_values('val_macro_AP', ascending=False).reset_index(drop=True)
print('\nT5 ensemble candidates (validation only, top 12):'); print(cand.head(12).to_string(index=False))
cand.to_csv(f'{PROJECT_DIR}/results/tables/T5_ensemble_candidates.csv', index=False)

best_ap = cand.val_macro_AP.max()
near = cand[cand.val_macro_AP >= best_ap - 0.002].sort_values(['n_models', 'val_macro_AP'], ascending=[True, False])
chosen = near.iloc[0]
final_runs = chosen['runs'].split('+')
print(f"\nCHOSEN: {final_runs}  (val Macro AP {chosen['val_macro_AP']}; best possible {best_ap}; "
      'rule: the simplest candidate within 0.002 AP of the best)')

# F10 correlation between the top runs (mean over labels, and for threat)
def corr_mat(cols):
    return np.mean([np.corrcoef(np.vstack([P[r][:, i] for r in top])) for i in cols], axis=0)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, (title, cols) in zip(ax, [('mean over 6 labels', range(6)), ('threat', [3])]):
    sns.heatmap(pd.DataFrame(corr_mat(cols), index=top, columns=top), annot=True, fmt='.2f', cmap='viridis', vmin=0.5, vmax=1, ax=a)
    a.set_title(f'Pearson correlation: {title}')
plt.suptitle('F10: how similar are the predictions of different runs?'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F10_prediction_correlation.png', dpi=150); plt.show()

# freeze: thresholds on validation, config, plots
Pf = np.mean([P[r] for r in final_runs], axis=0)
thr = tune_thresholds(Yva, Pf)
rep = per_label_report(Yva, Pf, thr)
print('\nT6 thresholds and validation results of the final system:'); print(rep.to_string(index=False))
rep.to_csv(f'{PROJECT_DIR}/results/tables/T6_final_thresholds_val.csv', index=False)
grid = np.arange(0.05, 0.951, 0.01)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, (l, a) in enumerate(zip(LABELS, axes.ravel())):
    a.plot(grid, [f1_score(Yva[:, i], Pf[:, i] >= t, zero_division=0) for t in grid]); a.axvline(thr[l], color='red', ls='--')
    a.set_title(f'{l} (threshold {thr[l]})'); a.set_xlabel('threshold'); a.set_ylabel('validation F1')
plt.suptitle('F11: F1 versus threshold (validation)'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F11_threshold_curves.png', dpi=150); plt.show()

cfg = {'runs': final_runs, 'thresholds': thr, 'val_macro_AP': round(macro_ap(Yva, Pf), 4),
       'val_macro_F1_tuned': round(rep.F1.mean(), 4), 'selected_on': 'validation only',
       'rule': 'simplest candidate within 0.002 val Macro AP of the best, must include a DistilBERT run',
       'frozen_at': time.strftime('%Y-%m-%d %H:%M')}
json.dump(cfg, open(f'{PROJECT_DIR}/results/final/final_config.json', 'w'), indent=2)
open(f'{PROJECT_DIR}/results/final/FROZEN.txt', 'w').write(f"Model and thresholds frozen at {cfg['frozen_at']}. Test sets may be evaluated now; do not change the choice.\n")
print('\nFROZEN:', cfg)
print('CELL 3 DONE')